# Problem 3: Attention in CNNs (ReducedMNIST + Spoken Digits)

We will **explain first, then implement**. The goal is to compare a plain CNN vs. a CNN with **spatial attention** and report accuracy + training time.

## Big idea (simple analogy)
Imagine you are reading a noisy page. A normal CNN looks everywhere equally. **Spatial attention** is like using a translucent highlighter: it fades unimportant areas and brightens the important strokes (e.g., the digit shape). This often improves accuracy, but it can add compute cost.

We will do this twice:
1. **ReducedMNIST images** (10 digits).
2. **Spoken digits** using **spectrogram images** (audio -> image).

We will keep the same CNN backbone for fair comparison. Only the attention module changes.

## Plan (what we will build)
1. Auto-detect dataset paths in this workspace.
2. Define **LeNet-style CNN** as baseline.
3. Add **Spatial Attention** (CBAM-style) to get the attention model.
4. Train both, measure time + accuracy, and compare in a table.
5. Write observations and future improvements.

In [10]:
# If needed, install dependencies (run once)
# !pip install torch torchvision torchaudio librosa soundfile pandas scikit-learn tqdm

In [11]:
import os
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import librosa
import pandas as pd
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

ModuleNotFoundError: No module named 'librosa'

In [5]:
# Auto-detect dataset paths
ROOT = Path(r"C:\\Users\\Antar\\NN_Assignments")

MNIST_CANDIDATES = [
    ROOT / "Assignment_2" / "ReducedMNIST_kaggle" / "Reduced MNIST Data",
    ROOT / "Assignment_1" / "Part_2" / "ReducedMNIST_kaggle" / "Reduced MNIST Data",
]

AUDIO_CANDIDATES = [
    ROOT / "Assignment_2" / "Problem_4" / "audio-dataset",
]

def pick_existing(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

MNIST_ROOT = pick_existing(MNIST_CANDIDATES)
AUDIO_ROOT = pick_existing(AUDIO_CANDIDATES)

MNIST_ROOT, AUDIO_ROOT

(WindowsPath('C:/Users/Antar/NN_Assignments/Assignment_2/ReducedMNIST_kaggle/Reduced MNIST Data'),
 WindowsPath('C:/Users/Antar/NN_Assignments/Assignment_2/Problem_4/audio-dataset'))

If the paths are `None`, update `MNIST_CANDIDATES` or `AUDIO_CANDIDATES` above.

# Part (a) ReducedMNIST
We will load the dataset using `ImageFolder`. We resize to 32x32, normalize, and train.

In [6]:
# Hyperparameters (you can tune)
BATCH_SIZE = 64
EPOCHS = 10  # set to 15-20 for better results
LR = 1e-3
IMG_SIZE = 32

mnist_train_dir = MNIST_ROOT / "Reduced Training data"
mnist_test_dir = MNIST_ROOT / "Reduced Testing data"

mnist_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

mnist_train_ds = datasets.ImageFolder(mnist_train_dir, transform=mnist_transform)
mnist_test_ds = datasets.ImageFolder(mnist_test_dir, transform=mnist_transform)

mnist_train_loader = DataLoader(mnist_train_ds, batch_size=BATCH_SIZE, shuffle=True)
mnist_test_loader = DataLoader(mnist_test_ds, batch_size=BATCH_SIZE, shuffle=False)

len(mnist_train_ds), len(mnist_test_ds)

(10000, 2000)

## CNN Backbone (LeNet-style)
We use a small CNN so training is fast.
- Conv -> ReLU -> Pool
- Conv -> ReLU -> Pool
- FC -> FC

### Spatial Attention Module
We add a **spatial attention** map after the second convolution. It uses avg + max pooling across channels and learns a 2D mask to highlight important regions.

In [7]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Channel-wise pooling -> 2 feature maps
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        pooled = torch.cat([avg_map, max_map], dim=1)
        attn = self.sigmoid(self.conv(pooled))
        return x * attn

class LeNetBase(nn.Module):
    def __init__(self, use_attention=False, num_classes=10):
        super().__init__()
        self.use_attention = use_attention
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.attn = SpatialAttention() if use_attention else None
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        if self.use_attention:
            x = self.attn(x)
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [8]:
def train_one_model(model, train_loader, test_loader, epochs=10, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    start = time.perf_counter()
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    train_time = time.perf_counter() - start

    # Evaluate
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    acc = 100.0 * correct / total
    return acc, train_time

def run_pair(train_loader, test_loader, epochs, lr):
    base = LeNetBase(use_attention=False)
    attn = LeNetBase(use_attention=True)

    base_acc, base_time = train_one_model(base, train_loader, test_loader, epochs, lr)
    attn_acc, attn_time = train_one_model(attn, train_loader, test_loader, epochs, lr)

    return {
        "base_acc": base_acc,
        "base_time": base_time,
        "attn_acc": attn_acc,
        "attn_time": attn_time,
    }

In [9]:
# Train on ReducedMNIST
mnist_results = run_pair(mnist_train_loader, mnist_test_loader, EPOCHS, LR)
mnist_results

NameError: name 'DEVICE' is not defined

# Part (b) Spoken digits using spectrogram images
We convert each audio clip to a **Mel spectrogram** (a 2D image). Then we train the same CNN models.

In [ ]:
# Build a dataset from wav -> mel-spectrogram image
class SpectrogramDataset(torch.utils.data.Dataset):
    def __init__(self, folder, img_size=64):
        self.files = sorted([f for f in Path(folder).glob('*.wav')])
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fp = self.files[idx]
        # label is after last underscore (e.g., M16_3.wav -> 3)
        label = int(fp.stem.split('_')[-1])
        y, sr = librosa.load(fp, sr=None, mono=True)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        S_db = librosa.power_to_db(S, ref=np.max)

        # Normalize to [0, 1]
        S_db -= S_db.min()
        if S_db.max() > 0:
            S_db /= S_db.max()

        # Resize to fixed size
        S_tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        S_tensor = F.interpolate(S_tensor.unsqueeze(0), size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
        S_tensor = S_tensor.squeeze(0)
        return S_tensor, label

audio_train_dir = AUDIO_ROOT / 'Train'
audio_test_dir = AUDIO_ROOT / 'Test'

spec_train = SpectrogramDataset(audio_train_dir, img_size=64)
spec_test = SpectrogramDataset(audio_test_dir, img_size=64)

spec_train_loader = DataLoader(spec_train, batch_size=BATCH_SIZE, shuffle=True)
spec_test_loader = DataLoader(spec_test, batch_size=BATCH_SIZE, shuffle=False)

len(spec_train), len(spec_test)

In [ ]:
# Train on spectrograms
spec_results = run_pair(spec_train_loader, spec_test_loader, EPOCHS, LR)
spec_results

# Results table
We summarize accuracy and training time.

In [ ]:
df = pd.DataFrame([
    {"Task": "ReducedMNIST", "Model": "CNN", "Accuracy (%)": mnist_results['base_acc'], "Train Time (s)": mnist_results['base_time']},
    {"Task": "ReducedMNIST", "Model": "CNN + Spatial Attention", "Accuracy (%)": mnist_results['attn_acc'], "Train Time (s)": mnist_results['attn_time']},
    {"Task": "Spoken Digits (Spectrogram)", "Model": "CNN", "Accuracy (%)": spec_results['base_acc'], "Train Time (s)": spec_results['base_time']},
    {"Task": "Spoken Digits (Spectrogram)", "Model": "CNN + Spatial Attention", "Accuracy (%)": spec_results['attn_acc'], "Train Time (s)": spec_results['attn_time']},
])
df

# Discussion (fill after running)
- **Accuracy change:** Did attention improve accuracy? On what task?
- **Training time:** Attention adds a small conv layer. Expect slightly more time.
- **Why:** Attention highlights the digit strokes or dominant time-frequency areas.

## Insights
Write 2-3 insights based on your results. Example: *"Attention helps more when the background is noisy or the object is small."*

## Future improvements
- Try **channel attention** or **self-attention**.
- Deeper CNN backbone (e.g., ResNet-18).
- Data augmentation (rotation, time masking).
- Tune learning rate and epochs.